# 임베딩 파인튜닝을 키워서 다시 재기 (RunPod RTX PRO 4500)

[ADR-097](../docs/adr/ADR-097-fine-tune-at-scale-on-a-rented-gpu.md)의 실행 노트북입니다. **바는 ADR에 먼저 적었습니다.**

2026-10-16 Colab T4 실행은 배치 32·1 epoch였고, 인용 과제에서 OpenAI와 **구별되지 않았습니다**
(+0.0032 [−0.0115, +0.0188]). 32GB에서 그 두 제약을 풀고, 학습한 모델로 **말뭉치 37,854편을 인코딩**까지 합니다.

| 변형 | 무엇 | 역할 |
|---|---|---|
| `b256-e3` | 배치 256, 3 epoch | **주 후보** — 이것만 채택 판정 |
| `b32-e3` | 배치 32, 3 epoch | 귀속용 (epoch만 늘림) |
| `inbatch-2026-10-16` | 기존 HF 모델, 인코딩만 | 귀속용 (옛 모델의 자리) |

**판정은 여기서 하지 않습니다.** 마지막 셀이 올린 `.npy`를 저장소로 받아, ADR-039 자리(미래 과제 간선 0개 집단)에서
기존 하네스로 잽니다. 이 노트북 안의 인용 과제 점수는 **학습이 됐는지 보는 진단**입니다.

## RunPod 준비

1. **템플릿은 CUDA 12.8 이상 PyTorch**로 고르세요(예: `runpod/pytorch` 2.8 / CUDA 12.8). RTX PRO 4500은 Blackwell(sm_120)이라
   그보다 낮은 torch는 `no kernel image is available`로 죽습니다. 2절이 학습 전에 잡습니다.
2. **볼륨은 `/workspace`에 20GB 이상.** 체크포인트·모델·행렬이 모두 거기 남아 팟을 다시 켜도 이어서 돕니다.
3. 이 노트북 하나만 올리면 됩니다. 저장소를 clone하지 않습니다 — 데이터는 HF 데이터셋 저장소에서 받습니다.

## 실행 순서

`SMOKE = True`로 한 번 끝까지 돌려 코드가 도는지 보고, `False`로 본 실행을 하세요. SMOKE는 말뭉치를 줄여 재므로
**수치를 결과로 읽지 마세요.** SMOKE 산출물에는 `-smoke`가 붙고 판정 명령도 찍지 않습니다.

**토큰은 4절에서 둘 다 묻습니다**(HF 쓰기, DagsHub). 학습이 몇 시간 돈 뒤 입력 대기로 멈추지 않게 하려는 것입니다.

## 1. 설정 — 여기만 바꿉니다

In [ ]:
SMOKE = True                      # 먼저 True로 한 번, 그다음 False

WORK = "/workspace"               # RunPod 볼륨. 팟을 다시 켜도 남는 자리입니다
DATA_REPO_NAME = "arxiv-graphrag-colab"   # 계정명은 토큰에서 꺼냅니다
DATA_REPO_OWNER = None            # 다른 계정의 데이터 저장소일 때만

BASE_MODEL = "BAAI/bge-base-en-v1.5"
MAX_SEQ_LENGTH = 512              # 제목+초록 중앙값 258토큰. 줄이면 절단을 재게 됩니다

# ADR-097에 적은 변형이 전부입니다. 결과를 보고 늘리지 않습니다.
# 순서는 주 후보 먼저입니다 - 시간이 모자라 끊겨도 판정할 것이 남습니다.
VARIANTS = [
    {"name": "b256-e3", "batch_size": 256, "epochs": 3, "primary": True},
    {"name": "b32-e3", "batch_size": 32, "epochs": 3, "primary": False},
]
# 학습하지 않고 인코딩만 합니다. 2026-10-16 모델이 ADR-039 자리에서 어디인지 보려는 것입니다.
EXISTING = {"name": "inbatch-2026-10-16", "repo_suffix": "bge-arxiv-citation-inbatch",
            "expected_recall_at_10": 0.2051}

LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
SEED = 20261015
TOP_K = 10

# GradCache가 GPU에 한 번에 올리는 텍스트 수. 추측하지 않고 6절이 재서 고릅니다
# (T4에서 4로 추측했다가 87%를 놀렸고, 32는 OOM이었습니다).
MINI_BATCH_CANDIDATES = (32, 64, 128, 256)
MEMORY_HEADROOM = 0.75
MINI_BATCH = 32                   # GPU가 없을 때만 쓰는 값

ENCODE_BATCH = 256                # 인코딩 배치. OOM이면 절반씩 줄입니다

# 학습 곡선은 팟 밖에 남깁니다. 팟 안 CSV나 mlruns/는 팟을 지우면 함께 사라집니다.
TRACKING = "mlflow"               # "mlflow" | "tensorboard" | "off"(추적을 끕니다 - 곡선을 잃습니다)
MLFLOW_TRACKING_URI = "https://dagshub.com/rndigkwk/arxiv_graph_RAG.mlflow"
MLFLOW_USERNAME = "rndigkwk"
MLFLOW_EXPERIMENT = "arxiv-graphrag-embedding"

HF_REPO_PREFIX = "bge-arxiv"      # 모델 저장소: <계정>/bge-arxiv-<변형>, 비공개

SMOKE_CORPUS = 3000
SMOKE_PAIRS = 600
SMOKE_QUERIES = 40

## 2. GPU 확인

GPU가 없으면 멈춥니다(SMOKE는 CPU로도 돕니다). **작은 연산 하나를 먼저 돌립니다** — torch가 이 GPU의 아키텍처를
모르면 모델을 받은 뒤가 아니라 여기서 몇 초 만에 죽습니다.

In [ ]:
import os
import sys

# 조각화로 인한 OOM을 줄입니다. torch를 가져오기 전에 설정해야 효과가 있습니다.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch

print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.version.cuda)
HAS_GPU = torch.cuda.is_available()
if not HAS_GPU:
    print("GPU가 없습니다.")
    if not SMOKE:
        raise SystemExit("GPU 없이 학습을 시작하지 않습니다. SMOKE=True로 코드만 확인하세요.")
    BF16 = False
else:
    props = torch.cuda.get_device_properties(0)
    major, minor = torch.cuda.get_device_capability(0)
    arch = f"sm_{major}{minor}"
    print(f"gpu {props.name} | {props.total_memory / 1024**3:.1f} GiB | {arch}")
    if arch not in torch.cuda.get_arch_list():
        raise SystemExit(
            f"이 torch는 {arch}용 커널이 없습니다(지원: {torch.cuda.get_arch_list()}). "
            "CUDA 12.8 이상으로 빌드된 PyTorch 템플릿으로 팟을 다시 만드세요."
        )
    torch.zeros(8, device='cuda').sum().item()   # CUDA 컨텍스트가 실제로 서는지
    BF16 = torch.cuda.is_bf16_supported()
    print("bf16", BF16)

if TRACKING not in ("mlflow", "tensorboard", "off"):
    raise SystemExit(f"TRACKING 값이 잘못됐습니다: {TRACKING}")

## 3. 설치

`mlflow`가 아니라 **`mlflow-skinny`**입니다. 전체 mlflow는 Flask를 끌어오고, RunPod 이미지의 OS blinker를 pip이
지우지 못해 설치가 깨집니다(2026-10-22). torch는 건드리지 않습니다 — 템플릿의 torch가 이 GPU를 아는 판입니다.

In [ ]:
%pip install -q -U "sentence-transformers[train]>=5.1" "datasets>=2.19" accelerate tensorboard
%pip install -q -U mlflow-skinny

# 임포트까지 해 봅니다. pip은 의존성 충돌을 경고만 하고 넘어가서, 몇 셀 뒤에야 깨집니다.
import importlib.metadata as metadata

import accelerate  # noqa: F401
from datasets import Dataset  # noqa: F401
from sentence_transformers import SentenceTransformer  # noqa: F401
from sentence_transformers.training_args import BatchSamplers  # noqa: F401

for package in ("sentence-transformers", "transformers", "torch", "datasets", "accelerate",
                "huggingface_hub", "mlflow-skinny"):
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, "없음")

## 4. 토큰 — 학습 전에 전부 받습니다

HF 토큰(쓰기 권한)과 DagsHub 토큰입니다. 화면에 찍히지 않고 환경변수로만 둡니다. **노트북에 적지 마세요** — 커밋에
섞입니다. MLflow 인증도 여기서 확인합니다. 학습을 시작한 뒤에 깨지면 그 사실을 몇 시간 뒤에 압니다.

In [ ]:
import getpass

from huggingface_hub import HfApi

token = os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face 토큰 (쓰기 권한): ")
HF_USER = HfApi(token=token).whoami()["name"]
DATA_REPO = f"{DATA_REPO_OWNER or HF_USER}/{DATA_REPO_NAME}"
print("HF 계정", HF_USER, "| 데이터 저장소", DATA_REPO)

# SMOKE는 서버에 남기지 않습니다. 몇 분짜리 점검이 실험 목록을 채우면 본 실행을 찾기 어렵습니다.
REMOTE_TRACKING = TRACKING == "mlflow" and not SMOKE
if REMOTE_TRACKING:
    os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
    os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
    if not os.environ.get("MLFLOW_TRACKING_PASSWORD"):
        os.environ["MLFLOW_TRACKING_PASSWORD"] = getpass.getpass("DagsHub 토큰: ")
    os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT
    import mlflow

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    try:
        found = mlflow.MlflowClient().search_experiments()
    except Exception as exc:
        raise SystemExit(f"MLflow 서버에 연결하지 못했습니다: {type(exc).__name__}: {str(exc)[:200]}") from exc
    print(f"MLflow 인증 OK ({len(found)}개 실험) -> {MLFLOW_TRACKING_URI}")
elif TRACKING == "mlflow":
    print("SMOKE라 MLflow에 남기지 않습니다.")

## 5. 데이터

`uv run python -m models.embedding_tuning.export_for_colab`이 만든 `exports/colab/`입니다. 2026-10-16 실행과 **같은 파일**이어야
비교가 됩니다. 자르는 선(학습 인용자 < 2026-08)이 미래 과제의 `feature_cut`과 같은지도 여기서 확인합니다.

In [ ]:
import gzip
import json
import random
from pathlib import Path

import numpy as np

random.seed(SEED)
np.random.seed(SEED)

WORK_DIR = Path(WORK)
DATA_DIR = WORK_DIR / "exports" / "colab"
REQUIRED = ("manifest.json", "corpus.jsonl.gz", "train_pairs.jsonl.gz",
            "eval_queries.jsonl.gz", "baseline_recall.jsonl.gz")
if not all((DATA_DIR / name).exists() for name in REQUIRED):
    from huggingface_hub import snapshot_download

    snapshot_download(DATA_REPO, repo_type="dataset", token=token, local_dir=str(WORK_DIR),
                      allow_patterns=["exports/colab/*"])
missing = [name for name in REQUIRED if not (DATA_DIR / name).exists()]
if missing:
    raise SystemExit(
        f"{DATA_REPO}에 exports/colab/{missing}가 없습니다. 저장소에서 "
        "`uv run python -m models.embedding_tuning.export_for_colab`로 만들어 올리세요."
    )


def read_jsonl(path):
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle]


manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
corpus_rows = read_jsonl(DATA_DIR / "corpus.jsonl.gz")
train_rows = read_jsonl(DATA_DIR / "train_pairs.jsonl.gz")
eval_rows = read_jsonl(DATA_DIR / "eval_queries.jsonl.gz")
baseline_rows = read_jsonl(DATA_DIR / "baseline_recall.jsonl.gz")

# 자르는 선이 미래 과제의 feature_cut(2026-08)과 같아야 학습이 미래를 보지 않습니다(ADR-097).
CUT_LINES = {"snapshot": "2026-10-14", "train_citers_before": "2026-08"}
if manifest["snapshot_version"] != CUT_LINES["snapshot"]:
    raise SystemExit(f"스냅샷이 다릅니다: {manifest['snapshot_version']}")
if manifest["split"]["test_citers_from"] != CUT_LINES["train_citers_before"]:
    raise SystemExit(f"자르는 선이 다릅니다: {manifest['split']['test_citers_from']}")
if len(corpus_rows) != 37854:
    raise SystemExit(f"말뭉치가 37,854편이 아닙니다: {len(corpus_rows)}")
print("snapshot", manifest["snapshot_version"], "| corpus", len(corpus_rows),
      "| train pairs", len(train_rows), "| eval queries", len(eval_rows))

In [ ]:
# SMOKE에서는 말뭉치·쌍·질의를 줄입니다. 후보 수가 달라지므로 수치를 비교에 쓸 수 없습니다.
if SMOKE:
    keep_ids = {row["anchor"] for row in eval_rows[:SMOKE_QUERIES]}
    for row in eval_rows[:SMOKE_QUERIES]:
        keep_ids.update(row["gold"])
    extra = [row["arxiv_id"] for row in corpus_rows if row["arxiv_id"] not in keep_ids]
    keep_ids.update(extra[: max(0, SMOKE_CORPUS - len(keep_ids))])
    corpus_rows = [row for row in corpus_rows if row["arxiv_id"] in keep_ids]
    eval_rows = eval_rows[:SMOKE_QUERIES]
    train_rows = [row for row in train_rows
                  if {row["anchor"], row["positive"]} <= keep_ids][:SMOKE_PAIRS]
    print("SMOKE: corpus", len(corpus_rows), "pairs", len(train_rows), "queries", len(eval_rows))
    print("경고: 후보 수를 줄였으므로 아래 수치를 결과로 읽지 마세요.")

text_of = {row["arxiv_id"]: row["text"] for row in corpus_rows}
corpus_ids = [row["arxiv_id"] for row in corpus_rows]
position = {arxiv_id: index for index, arxiv_id in enumerate(corpus_ids)}
baseline_recall = {row["anchor"]: row["recall_at_10"] for row in baseline_rows}
queries = [(row["anchor"], [p for p in row["gold"] if p in position])
           for row in eval_rows if row["anchor"] in position]
queries = [(anchor, gold) for anchor, gold in queries if gold]
openai_recall = np.asarray([baseline_recall[anchor] for anchor, _ in queries])
print("질의", len(queries), "| OpenAI recall@10", round(float(openai_recall.mean()), 4))

## 6. 도구 — 인코딩, 채점, 쌍체 부트스트랩

채점은 2026-10-16 노트북과 같습니다 — 질의 논문 자신을 후보에서 빼고 전체 말뭉치를 순위 매깁니다.
`normalize_embeddings=True`와 `float32`는 저장소 하네스와 **같아야** 합니다.

In [ ]:
import gc
import hashlib
import time


def free_gpu(*objects):
    for item in objects:
        try:
            item.to("cpu")
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def encode_texts(model, texts, size=ENCODE_BATCH):
    """OOM이면 배치를 절반으로 줄여 다시 시도합니다. 터지는 대신 느려집니다."""
    while True:
        try:
            vectors = model.encode(texts, batch_size=size, normalize_embeddings=True,
                                   convert_to_numpy=True, show_progress_bar=False)
            return np.asarray(vectors, dtype=np.float32), size
        except torch.cuda.OutOfMemoryError:
            if size <= 1:
                raise
            torch.cuda.empty_cache()
            size //= 2
            print(f"  OOM - 인코딩 배치를 {size}로 줄입니다", flush=True)


def encode_corpus(model):
    vectors, _ = encode_texts(model, [text_of[arxiv_id] for arxiv_id in corpus_ids])
    return vectors


def per_query_recall(vectors, top_k=TOP_K):
    rows = np.stack([vectors[position[anchor]] for anchor, _ in queries])
    similarity = rows @ vectors.T
    for index, (anchor, _) in enumerate(queries):
        similarity[index, position[anchor]] = -np.inf
    ranked = np.argpartition(-similarity, top_k, axis=1)[:, :top_k]
    recalls, hits, reciprocal = [], [], []
    for index, (_, gold) in enumerate(queries):
        order = ranked[index][np.argsort(-similarity[index, ranked[index]])]
        retrieved = [corpus_ids[spot] for spot in order]
        found = [rank for rank, paper in enumerate(retrieved, start=1) if paper in set(gold)]
        recalls.append(len(set(gold) & set(retrieved)) / len(gold))
        hits.append(float(bool(found)))
        reciprocal.append(1.0 / found[0] if found else 0.0)
    return np.asarray(recalls), np.asarray(hits), np.asarray(reciprocal)


def metrics(vectors):
    recalls, hits, reciprocal = per_query_recall(vectors)
    return {
        "queries": len(queries),
        f"recall_at_{TOP_K}": round(float(recalls.mean()), 4),
        f"hit_at_{TOP_K}": round(float(hits.mean()), 4),
        "mrr": round(float(reciprocal.mean()), 4),
    }, recalls


def paired_bootstrap(baseline, candidate, samples=2000):
    difference = candidate - baseline
    generator = np.random.default_rng(SEED)
    indices = generator.integers(0, len(difference), size=(samples, len(difference)))
    low, high = np.percentile(difference[indices].mean(axis=1), [2.5, 97.5])
    return {"mean_difference": round(float(difference.mean()), 4),
            "ci95_low": round(float(low), 4), "ci95_high": round(float(high), 4),
            "excludes_zero": bool(low > 0.0 or high < 0.0)}


def vram_peak_gb():
    return round(torch.cuda.max_memory_allocated() / 1024**3, 2) if torch.cuda.is_available() else 0.0


SUFFIX = "-smoke" if SMOKE else ""
RUNS = WORK_DIR / "runs"
OUT = WORK_DIR / "encoded"
RUNS.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

## 7. 튜닝 전 기준선

SMOKE가 아니면 recall@10이 **0.1909** 근처여야 합니다(2026-10-16 값). 크게 다르면 토크나이저·정규화·질의 중 하나가
어긋난 것이고, 그때는 아래 비교가 모두 성립하지 않습니다.

In [ ]:
untuned = SentenceTransformer(BASE_MODEL)
untuned.max_seq_length = MAX_SEQ_LENGTH
start = time.perf_counter()
untuned_metrics, untuned_recall = metrics(encode_corpus(untuned))
print("untuned", untuned_metrics, f"| 인코딩 {(time.perf_counter() - start) / 60:.1f}분")
if not SMOKE:
    gap = abs(untuned_metrics[f"recall_at_{TOP_K}"] - 0.1909)
    print("기대값 0.1909과의 차이", round(gap, 4), "-> 재현됨" if gap < 0.01 else "-> 어긋남. 원인부터 찾으세요")
free_gpu(untuned)
del untuned

## 8. MINI_BATCH 재기

손실의 배치(배치 안 음성 수)와 GPU에 한 번에 올리는 양을 나눕니다. 가장 큰 배치(256)로 몇 스텝씩 돌려
실제 최대 메모리를 보고 고릅니다. 고른 값은 두 변형에 같이 씁니다 — 결과는 바뀌지 않고 속도만 바뀝니다.

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss


def pair_dataset(rows):
    return Dataset.from_dict({"anchor": [text_of[row["anchor"]] for row in rows],
                              "positive": [text_of[row["positive"]] for row in rows]})


def probe_mini_batch(steps=3):
    if not HAS_GPU:
        print("GPU 없음 -> MINI_BATCH", MINI_BATCH)
        return MINI_BATCH
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    batch = max(variant["batch_size"] for variant in VARIANTS)
    probe = pair_dataset(train_rows[: batch * (steps + 1)])
    chosen = None
    for candidate in MINI_BATCH_CANDIDATES:
        model = SentenceTransformer(BASE_MODEL)
        model.max_seq_length = MAX_SEQ_LENGTH
        torch.cuda.reset_peak_memory_stats()
        try:
            SentenceTransformerTrainer(
                model=model,
                args=SentenceTransformerTrainingArguments(
                    output_dir=str(RUNS / f"probe-{candidate}"), max_steps=steps,
                    per_device_train_batch_size=batch, learning_rate=LEARNING_RATE,
                    bf16=BF16, fp16=not BF16, seed=SEED, logging_strategy="no",
                    save_strategy="no", report_to=[],
                ),
                train_dataset=probe,
                loss=CachedMultipleNegativesRankingLoss(model, mini_batch_size=candidate),
            ).train()
            peak = torch.cuda.max_memory_allocated() / 1024**3
            fits = peak <= total * MEMORY_HEADROOM
            print(f"  mini_batch {candidate:3d} -> 최대 {peak:.2f} / {total:.1f} GiB {'(여유)' if fits else '(한도 초과)'}")
            if fits:
                chosen = candidate
            free_gpu(model)
            del model
            if not fits:
                break
        except torch.cuda.OutOfMemoryError:
            print(f"  mini_batch {candidate:3d} -> OOM")
            free_gpu(model)
            del model
            break
    return chosen or MINI_BATCH_CANDIDATES[0]


MINI_BATCH = probe_mini_batch()
print("MINI_BATCH =", MINI_BATCH)

## 9. 학습

**epoch를 고르지 않습니다.** 최종 모델은 마지막 epoch 끝 모델입니다. epoch마다 인용 과제 점수를 찍지만 진단이고,
그것을 보고 중간 모델을 고르면 평가 질의로 모델을 고른 것이 됩니다.

**저장된 모델은 설정이 같을 때만 다시 씁니다.** 디렉터리 이름에 설정 지문을 넣고 `trained_on.json`을 대조합니다 —
SMOKE가 저장한 모델을 본 실행이 "저장된 모델"로 읽어 학습을 건너뛰는 일을 막습니다(2026-10-22에 그럴 뻔했습니다).
체크포인트가 있으면 거기서 이어서 돕니다.

In [ ]:
from transformers import TrainerCallback
from transformers.trainer_utils import get_last_checkpoint


class EpochScore(TrainerCallback):
    """epoch 끝마다 인용 과제 점수를 찍고 추적 서버에 남깁니다. 진단용입니다."""

    def __init__(self):
        self.rows = []

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        row, _ = metrics(encode_corpus(model))
        row = {"epoch": round(float(state.epoch or 0.0), 2), "step": state.global_step, **row}
        self.rows.append(row)
        print("  epoch", row, flush=True)
        if REMOTE_TRACKING:
            import mlflow

            if mlflow.active_run():
                mlflow.log_metrics({f"diag_{key}": value for key, value in row.items()
                                    if key.startswith(("recall", "hit", "mrr"))},
                                   step=state.global_step)
        model.train()


def fingerprint(variant):
    config = {
        "base": BASE_MODEL, "max_seq": MAX_SEQ_LENGTH, "batch": variant["batch_size"],
        "epochs": variant["epochs"], "lr": LEARNING_RATE, "warmup": WARMUP_RATIO, "seed": SEED,
        "pairs": len(train_rows), "sampler": "no_duplicates", "smoke": bool(SMOKE),
    }
    return config, hashlib.sha1(json.dumps(config, sort_keys=True).encode()).hexdigest()[:8]


def tracking_arguments(variant):
    if SMOKE or TRACKING == "off":
        return {"report_to": []}
    if TRACKING == "tensorboard":
        return {"report_to": ["tensorboard"], "logging_dir": str(RUNS / f"tb-{variant['name']}")}
    return {"report_to": ["mlflow"], "run_name": f"runpod-{variant['name']}"}


def train(variant):
    config, fp = fingerprint(variant)
    saved = RUNS / f"out-{variant['name']}-{fp}"
    stamp = saved / "trained_on.json"
    if stamp.exists():
        _have = json.loads(stamp.read_text(encoding="utf-8"))["config"]
        _want = config
        if _have != _want:
            raise SystemExit(f"{saved}의 학습 내역이 설정과 다릅니다. 지우고 다시 돌리세요.")
        print(f"{variant['name']}: 저장된 모델을 다시 씁니다 ({saved})")
        return saved, json.loads(stamp.read_text(encoding="utf-8"))

    model = SentenceTransformer(BASE_MODEL)
    model.max_seq_length = MAX_SEQ_LENGTH
    steps_per_epoch = max(1, len(train_rows) // variant["batch_size"])
    ckpt_dir = RUNS / f"ckpt-{variant['name']}-{fp}"
    arguments = SentenceTransformerTrainingArguments(
        output_dir=str(ckpt_dir),
        num_train_epochs=variant["epochs"],
        per_device_train_batch_size=variant["batch_size"],
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        bf16=BF16, fp16=HAS_GPU and not BF16,
        seed=SEED,
        # 한 앵커가 참고문헌 여러 편과 쌍을 이룹니다. 배치가 크면 같은 앵커의 다른 양성이
        # 음성으로 들어오므로, 같은 텍스트가 한 배치에 두 번 들어가지 않게 합니다(ADR-097).
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        logging_steps=max(1, steps_per_epoch // 20),
        save_strategy="steps",
        save_steps=max(1, steps_per_epoch // 2),
        save_total_limit=2,
        **tracking_arguments(variant),
    )
    scores = EpochScore()
    trainer = SentenceTransformerTrainer(
        model=model, args=arguments, train_dataset=pair_dataset(train_rows),
        loss=CachedMultipleNegativesRankingLoss(model, mini_batch_size=MINI_BATCH),
        callbacks=[scores],
    )
    checkpoint = get_last_checkpoint(str(ckpt_dir)) if ckpt_dir.exists() else None
    if checkpoint:
        print(f"{variant['name']}: 체크포인트에서 이어서 학습합니다 ({checkpoint})")
    if HAS_GPU:
        torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    trainer.train(resume_from_checkpoint=checkpoint)
    minutes = (time.perf_counter() - start) / 60
    model.save(str(saved))
    losses = [row["loss"] for row in trainer.state.log_history if "loss" in row]
    record = {
        "config": config, "fingerprint": fp, "resumed_from": checkpoint,
        "train_minutes": round(minutes, 1),
        "train_pairs_per_second": round(len(train_rows) * variant["epochs"] / max(minutes * 60, 1e-9), 1),
        "vram_peak_gb": vram_peak_gb(), "mini_batch_used": MINI_BATCH,
        "steps": trainer.state.global_step,
        "loss": {"first": losses[0], "last": losses[-1], "min": min(losses)} if losses else {},
        "epoch_scores": scores.rows,
    }
    stamp.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    print(variant["name"], {k: record[k] for k in ("train_minutes", "train_pairs_per_second", "vram_peak_gb")})
    free_gpu(model)
    del model, trainer
    return saved, record


trained = {}
for variant in VARIANTS:
    trained[variant["name"]] = train(variant)
print("학습 완료", list(trained))

## 10. 인코딩 — 저장소 판정에 넘길 행렬

학습한 두 모델과 2026-10-16 모델로 말뭉치를 인코딩합니다. **행 순서는 `corpus.jsonl.gz` 그대로**이고, 그것이
파생 캐시(`embedding_ids.json`)와 같은 순서입니다. 행 수가 말뭉치와 다르면 저장하지 않습니다 — 어긋난 행렬은 조용히
다른 논문의 벡터로 점수를 냅니다.

2026-10-16 모델은 인용 과제에서 **0.2051**이 다시 나와야 합니다. 그것이 이 노트북의 채점이 그때와 같다는 확인입니다.

In [ ]:
encoded = {}


def encode_and_save(name, model_source, record=None, expected=None):
    model = SentenceTransformer(str(model_source), token=token)
    model.max_seq_length = MAX_SEQ_LENGTH
    if HAS_GPU:
        torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    matrix = encode_corpus(model)
    minutes = (time.perf_counter() - start) / 60
    if matrix.shape[0] != len(corpus_ids):
        raise SystemExit(f"{name}: 행 수가 다릅니다 {matrix.shape[0]} != {len(corpus_ids)}")
    row, recall = metrics(matrix)
    if expected is not None and not SMOKE:
        gap = abs(row[f"recall_at_{TOP_K}"] - expected)
        print(f"  기대값 {expected}과의 차이 {gap:.4f}", "-> 재현됨" if gap < 0.005 else "-> 어긋남")
    stem = f"local-bge-arxiv-{name}{SUFFIX}"
    np.save(OUT / f"{stem}.npy", matrix)
    card = {
        "model": str(model_source), "variant": name, "rows": int(matrix.shape[0]),
        "dimensions": int(matrix.shape[1]), "max_seq_length": MAX_SEQ_LENGTH,
        "normalize_embeddings": True, "encode_minutes": round(minutes, 1),
        "vram_peak_gb": vram_peak_gb(), "smoke": bool(SMOKE),
        "cut_lines": CUT_LINES, "citation_task": row, "training": record,
    }
    (OUT / f"{stem}.json").write_text(json.dumps(card, ensure_ascii=False, indent=2), encoding="utf-8")
    print(name, row, f"| 인코딩 {minutes:.1f}분 | {stem}.npy")
    encoded[name] = (row, recall, card)
    free_gpu(model)
    del model


for variant in VARIANTS:
    saved, record = trained[variant["name"]]
    encode_and_save(variant["name"], saved, record)
encode_and_save(EXISTING["name"], f"{HF_USER}/{EXISTING['repo_suffix']}",
                expected=EXISTING["expected_recall_at_10"])

## 11. 보고서 — 진단 수치

인용 과제(365질의)입니다. **판정 자리가 아닙니다.** 학습이 됐는지(`vs_untuned`)와 2026-10-16 대비 어디인지를 봅니다.

In [ ]:
results = {
    "text-embedding-3-large": {"recall_at_10": round(float(openai_recall.mean()), 4)},
    f"{BASE_MODEL} (untuned)": {**untuned_metrics,
                                 "vs_openai": paired_bootstrap(openai_recall, untuned_recall)},
}
for name, (row, recall, card) in encoded.items():
    results[name] = {**row, "vs_openai": paired_bootstrap(openai_recall, recall),
                     "vs_untuned": paired_bootstrap(untuned_recall, recall),
                     "training": card["training"]}
report = {
    "adr": "ADR-097", "smoke": bool(SMOKE), "snapshot_version": manifest["snapshot_version"],
    "cut_lines": CUT_LINES, "base_model": BASE_MODEL, "max_seq_length": MAX_SEQ_LENGTH,
    "primary": next(v["name"] for v in VARIANTS if v["primary"]),
    "gpu": torch.cuda.get_device_name(0) if HAS_GPU else None,
    "corpus": len(corpus_ids), "train_pairs": len(train_rows), "queries": len(queries),
    "note": "인용 과제 진단입니다. 판정은 저장소에서 ADR-039 자리(미래 과제 간선 0개 집단)로 합니다.",
    "results": results,
}
REPORT_PATH = OUT / f"finetune_scale_report{SUFFIX}.json"
REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(json.dumps({name: {k: v for k, v in value.items() if k != "training"}
                  for name, value in results.items()}, ensure_ascii=False, indent=2))

## 12. 올리기 — HF를 거칩니다

모델은 변형마다 **비공개** 모델 저장소에, 행렬·카드·보고서는 데이터셋 저장소에 올립니다. 행렬을 MLflow에도 올리지
않습니다 — 두 곳에 두면 둘이 갈라집니다. TensorBoard를 썼다면 이벤트 파일도 모델 저장소에 함께 올라갑니다.

SMOKE 산출물은 이름에 `-smoke`가 붙어 올라가고, 판정 명령은 찍지 않습니다.

In [ ]:
api = HfApi(token=token)
api.create_repo(DATA_REPO, repo_type="dataset", private=True, exist_ok=True)
uploaded = []
for path in sorted(OUT.glob(f"*{SUFFIX}.*")):
    if SMOKE != path.stem.endswith("-smoke"):
        continue
    api.upload_file(path_or_fileobj=str(path), path_in_repo=path.name,
                    repo_id=DATA_REPO, repo_type="dataset")
    uploaded.append(path.name)
    print("올림:", path.name, f"{path.stat().st_size / 1e6:.1f}MB")

if not SMOKE:
    for variant in VARIANTS:
        saved, _ = trained[variant["name"]]
        repo = f"{HF_USER}/{HF_REPO_PREFIX}-{variant['name']}"
        api.create_repo(repo, private=True, exist_ok=True)
        api.upload_folder(repo_id=repo, folder_path=str(saved))
        tb_dir = RUNS / f"tb-{variant['name']}"
        if tb_dir.exists():
            api.upload_folder(repo_id=repo, folder_path=str(tb_dir), path_in_repo="tensorboard")
        print("모델:", repo)

matrices = [name for name in uploaded if name.startswith("local-")]
if SMOKE:
    print("\nSMOKE 산출물입니다. 이 파일로 판정하지 마세요. SMOKE=False로 본 실행을 하세요.")
else:
    # 받는 쪽은 보통 Windows PowerShell입니다. bash 줄바꿈이 깨지므로 명령은 한 줄씩 찍습니다.
    print("\n저장소에서 받을 명령(한 줄입니다):")
    print(f"hf download {DATA_REPO} {' '.join(matrices)} --repo-type dataset --local-dir derived/2026-10-14")
    print("\n판정 명령(변형마다 두 줄):")
    for name in [v["name"] for v in VARIANTS] + [EXISTING["name"]]:
        slug = name.replace("-", "_")
        print(f"uv run python -m evals.related_study --cases evals/cases_related_future.json --variants vector --embeddings derived/2026-10-14/local-bge-arxiv-{name}.npy --output reports/embedding_ft_{slug}.json")
        print(f"uv run python -m evals.embedding_slice_study --baseline reports/embedding_openai.json --candidate reports/embedding_ft_{slug}.json --output reports/embedding_slice_ft_{slug}.json")
del token

## 13. 판정 — 저장소에서

받은 행렬로 위 명령을 돌리면 `reports/embedding_slice_ft_*.json`의 `by_degree_slice.no_citation_edges`가 판정 자리입니다.
ADR-097에 적은 바 그대로입니다 — **`b256-e3`만 판정합니다.**

| 지표 (간선 0개, 1,112 시드) | OpenAI | 튜닝 전 bge-base | 넘어야 하는 선 |
|---|---:|---:|---|
| recall@5 | 0.1227 | 0.1124 | 평균 차 양수, 구간이 0 제외 |
| hit@5 | 0.4739 | 0.4397 | 구간 전체가 0 아래이면 안 됨 |
| MRR | 0.3457 | 0.3182 | 구간 전체가 0 아래이면 안 됨 |

부가 조건: 간선 1~4개 집단(251 시드, OpenAI 0.2154)에서 recall@5 구간 전체가 0 아래이면 안 됩니다.
**동점은 통과가 아닙니다.** 상한은 0.783이고, OpenAI보다 +0.03 넘게 오르면 행 순서부터 의심합니다.

## 팟을 끄기 전에

`encoded/`와 모델 저장소가 올라갔는지 HF에서 확인하고 끄세요. 볼륨을 남기면 `runs/`의 체크포인트로 이어서 돌 수 있습니다.